# COMP0178 — Lesson 3a: Database Storage

- Source: [Lecture 3a: Database Storage](../Lectures/Lecture_3a.pdf), all **24 PDF pages**. References use PDF page order; printed slide numbering skips a number after page 11.
- Goal: explain how records reach memory, how indexes avoid unnecessary reads, and when hashing is appropriate.
- Original slide visuals are included. Examples, calculation exercises and SQL practice are supplementary teaching material, not additional claims made by the slides.
- Past-paper examples use the supplied 2023–2025 papers, not predictions or an official mark scheme.

## 1. Why storage organisation matters — pages 1–3

- A database must store its contents somewhere and retrieve the required records efficiently.
- A query asking for one row should not have to read an entire large table if an appropriate access structure exists.
- DBMS storage overlaps with operating-system concepts. A DBMS may manage its own files, buffers and replacement policies rather than relying entirely on generic OS behaviour.
- Keep logical tables separate from their physical organisation: a SQL row is a logical record; the DBMS places records in physical pages.

## 2. Storage hierarchy — pages 4–5

| Storage role | Example | Main purpose |
| --- | --- | --- |
| Primary | Main memory / RAM | Fast working data; normally volatile |
| Secondary | HDD or SSD | Persistent database files; slower access than RAM |
| Tertiary | Archive media, e.g. tape | Infrequently accessed historical data |

- Volatile means contents are lost without power; nonvolatile storage retains data but can still fail.
- SSDs use flash rather than rotating magnetic disks. The slides' relative speed/cost discussion is conceptual, not current product pricing.
- A table can be larger than available RAM. Queries load the needed portions into memory, which makes transfer costs important.

## 3. Blocks, pages and records — pages 6–8

- A **block** is a fixed-size transfer/allocation unit in the lecture's storage model. A **page** is a fixed-size unit managed in memory; DBMS literature often uses the terms interchangeably.
- The slide gives a rough 4–8 KB-scale example; distinguish bytes (B) from bits (b), and use the stated page size in calculations. Hardware sectors, OS pages and DBMS pages need not be identical.
- Several small records normally fit in a page. Large objects may span pages or be stored separately.
- Fixed-length records simplify locating slots and reusing deleted space. A free list records available locations; arbitrary reuse is easiest when physical order need not be maintained.
- Variable-length fields save unnecessary padding but require metadata and more careful free-space management.

**Supplementary capacity calculation:** assuming 4,096 usable bytes and 100-byte unspanned records, one page holds floor(4096/100) = 40 records. 1,000 records need ceil(1000/40) = 25 pages. Account for headers/slot metadata when provided; do not silently assume all page bytes are available.

In [ ]:
from math import ceil
page_bytes, header_bytes, record_bytes = 4096, 96, 100
records_per_page = (page_bytes - header_bytes) // record_bytes
pages_for_1000 = ceil(1000 / records_per_page)
print("Records per page:", records_per_page)  # 40
print("Pages for 1,000 records:", pages_for_1000)  # 25
assert (records_per_page, pages_for_1000) == (40, 25)

## 4. Variable-length records and slotted pages — page 9

- Store record location and length in metadata rather than assuming every record has the same size.
- A **slotted page** has a header/slot directory at one end and record data growing from the other. Free space lies between them.
- A slot can record an offset and length. If a record moves during compaction, its slot entry can be updated instead of requiring every reference to know the new physical offset.
- Deletion creates space that may be reused or compacted. Slot metadata itself consumes page capacity.

**Original lecture visual — PDF page 9**

[![Slotted-page explanation: metadata and variable-length record data share the page.](../assets/slide-images/Lecture_3a-page-009.png)](../assets/slide-images/Lecture_3a-page-009.png)

Slotted-page explanation: metadata and variable-length record data share the page.

Source: [Lecture_3a.pdf, PDF page 9](../Lectures/Lecture_3a.pdf#page=9).

## 5. Buffers, dirty pages and flushing — pages 10–11

- A **buffer** is temporary storage used while data is transferred or processed. In this lecture it holds copies outside the main persistent database; the usual DBMS buffer pool is in RAM.
- Reading a page loads a copy into a buffer. Reusing a resident copy can avoid another storage read.
- A **dirty page** has been changed in memory but not yet written back to its persistent data page.
- Before reusing a dirty buffer for another page, the DBMS must preserve the changes according to its recovery protocol; a clean page needs no write-back.
- **Flush** means write buffered changes to storage. Qualification: flushing does not inherently evict the page or make the buffer empty; a clean cached copy can remain available. The slide loosely combines flushing and clearing.
- A buffer write is not by itself the same as committing a transaction. Logging and recovery rules are later topics.

**Worked flow:** load page → change a salary → mark page dirty → flush under the DBMS protocol → cached page becomes clean. The page may remain in memory for another query.

## 6. Indexes and why scans are costly — pages 12–14

- A table scan examines its pages without a selective search structure. Under a cold, full-page-scan assumption, scanning B pages requires B page reads.
- An **index** maps search-key values to data locations, analogous to finding a topic through a book's index.
- A search key is an indexed attribute set; it need not be a unique/candidate/primary key.
- Indexes use storage and must be maintained during inserts, updates and deletes. They are useful when their saved reads justify this cost.
- Index entries can identify records, data pages, or—in some DBMS layouts—keys used for another lookup. Do not assume every pointer is a raw memory address.

## 7. Clustered and nonclustered indexes — page 15

- A **clustered** organisation orders/groups the data records by its search key. Range access can then follow nearby ordered pages.
- A table has one principal physical ordering, so it cannot independently be clustered by several unrelated orders.
- A **secondary/nonclustered index** is an additional access structure whose order differs from the table's data organisation.
- The clustered key is often the primary key, but those concepts are not synonymous. A primary key is a logical uniqueness rule.
- The slide's primary/secondary-index terminology describes its storage model; implementation terminology varies among DBMSs.

**Example:** data ordered by staffNo can have a secondary salary index. Finding salary=30000 through that index does not mean the table itself is stored in salary order.

**Original lecture visual — PDF page 15**

[![Clustered data order versus an additional secondary index.](../assets/slide-images/Lecture_3a-page-015.png)](../assets/slide-images/Lecture_3a-page-015.png)

Clustered data order versus an additional secondary index.

Source: [Lecture_3a.pdf, PDF page 15](../Lectures/Lecture_3a.pdf#page=15).

## 8. Dense and sparse indexes — page 16

- **Dense:** represents every search-key value, with enough references to retrieve all matching records. A repeated key can use a list of matching record references.
- **Sparse:** represents selected search-key values, often one entry per data block. Locate the appropriate preceding/bounding entry, then search the ordered data region.
- Sparseness relies on the corresponding order. In the lecture's basic model, a secondary index must cover all search-key values because omitted values cannot be recovered by following the data's unrelated order.
- Do not confuse a dense index with a unique index. Density is about coverage, not uniqueness.

**Example:** blocks start with keys 10, 30 and 50. To find 37 through a sparse index, choose the block beginning at 30, then search that block (and follow the storage convention if duplicates span blocks).

**Original lecture visual — PDF page 16**

[![Dense coverage versus sparse entries into ordered data blocks.](../assets/slide-images/Lecture_3a-page-016.png)](../assets/slide-images/Lecture_3a-page-016.png)

Dense coverage versus sparse entries into ordered data blocks.

Source: [Lecture_3a.pdf, PDF page 16](../Lectures/Lecture_3a.pdf#page=16).

## 9. Retrieval and multilevel indexes — pages 17–18

1. Search the index for the key or appropriate range.
2. Obtain the data-page/record reference.
3. Read the needed page if it is not already buffered.
4. Extract the required record(s).

- A small index may remain in memory. If it is too large, index the index: an upper level points to pages in the next level.
- Repeating this produces a tree. Keeping the root buffered saves a read on later searches.
- Count only storage reads actually needed under stated cache assumptions, not every pointer followed as an automatic disk access.

**Original lecture visual — PDF page 18**

[![A multilevel index directs the search through successive index pages.](../assets/slide-images/Lecture_3a-page-018.png)](../assets/slide-images/Lecture_3a-page-018.png)

A multilevel index directs the search through successive index pages.

Source: [Lecture_3a.pdf, PDF page 18](../Lectures/Lecture_3a.pdf#page=18).

## 10. B+ trees — page 19

- A B+ tree is balanced: root-to-leaf paths have the same depth. Nodes have many children, giving high **fan-out** and a shallow tree.
- Nodes are typically sized for storage pages, so following each level corresponds to a small number of page accesses.
- Internal nodes route searches using keys and child pointers. Leaves contain the entries used to reach data; linked leaves support ordered/range traversal.
- The slide shows a clustered variant with data records in leaves. In other B+ tree indexes, leaves contain keys and record references. State which layout is assumed.
- Inserts/deletes can split, merge or redistribute nodes to maintain balance. Do not claim that updates never require extra I/O.
- Lookup depth is logarithmic in the data size; exact page reads depend on fan-out, tree height, cached levels and whether a separate data page must be read.

**Supplementary read example:** root → internal page → leaf, root cached. A clustered leaf holding the desired row needs 2 uncached page reads. If the leaf only points to a separate uncached data page, add 1, giving 3. This is a stated example, not a universal B+ tree cost.

**Original lecture visual — PDF page 19**

[![A shallow B+ tree uses page-sized nodes and many child pointers.](../assets/slide-images/Lecture_3a-page-019.png)](../assets/slide-images/Lecture_3a-page-019.png)

A shallow B+ tree uses page-sized nodes and many child pointers.

Source: [Lecture_3a.pdf, PDF page 19](../Lectures/Lecture_3a.pdf#page=19).

## 11. Hashing and desirable hash functions — pages 20–21

- Hashing computes a bucket/location from the search key: `bucket = h(key)`.
- To look up that key, calculate the same function and search the relevant bucket.
- Desirable properties: deterministic; defined for every allowed key; fast; spreads the actual workload reasonably uniformly.
- Different keys may map to the same bucket: a **collision**. Hashing is not a guarantee of uniqueness.
- Example: `h(key)=key mod 4`. Keys 5, 9 and 13 all map to bucket 1. This toy function demonstrates mechanics; it may be poor for a skewed real workload.
- The output is a bucket/location identifier managed by the DBMS, not necessarily a directly usable CPU memory address.

## 12. Bucket overflow — page 22

- A bucket overflows when more records map to it than fit in its available capacity.
- **Chaining / open hashing / closed addressing:** attach overflow buckets or a linked structure to the home bucket. Follow that chain during lookup.
- **Open addressing / closed hashing:** find another slot using a probe sequence, such as linear probing. Lookup must follow the same sequence; deletion may need tombstones.
- The open/closed terminology is easy to reverse. Remember the mechanism rather than relying only on the word “open.”
- The slide highlights linked overflow storage as convenient for database deletion. Collision chains and skew can make operations much slower than the typical constant-time intuition.

**Original lecture visual — PDF page 22**

[![Overflow chaining versus probing for another available location.](../assets/slide-images/Lecture_3a-page-022.png)](../assets/slide-images/Lecture_3a-page-022.png)

Overflow chaining versus probing for another available location.

Source: [Lecture_3a.pdf, PDF page 22](../Lectures/Lecture_3a.pdf#page=22).

In [ ]:
# Supplementary mechanics: two distinct keys can share a bucket.
keys = [5, 9, 13, 2, 6]
buckets = {i: [] for i in range(4)}
for key in keys:
    buckets[key % 4].append(key)
print(buckets)
assert buckets[1] == [5, 9, 13]
assert buckets[2] == [2, 6]
# Locate 9 by the same hash function, then compare within that bucket.
assert 9 in buckets[9 % 4]

## 13. Sorted indexes versus hashing — pages 23–24

| Need | Sorted/B+ tree index | Hashing |
| --- | --- | --- |
| Equality lookup | Efficient | Typically very efficient with a good distribution |
| Range, e.g. rent 700–1400 | Ordered leaves are useful | Usually inspect many buckets; no useful key order |
| Sorted output | Can exploit matching index order | Extra sorting normally needed |
| Growth/skew | Balance-maintenance cost | Collision/overflow/resizing concerns |
| Cost shorthand | Logarithmic search depth | Expected constant bucket lookup under assumptions |

Do not read O(1) as guaranteed one physical read or O(log n) as a complete query-cost formula. Output size, caching and collisions still matter.

## 14. Runnable SQL practice — supplementary

Run setup in a Python 3 Jupyter kernel. The existing helper uses disposable in-memory **SQLite**, not a MySQL server. We inspect a plan to see an index used; this does not measure physical disk reads or prove that an index always wins.

In [ ]:
from pathlib import Path
import sys
practice_folder = next((p for p in [Path.cwd(), Path.cwd().parent,
    Path.cwd() / "COMP0178", Path("/Users/peidi/Desktop/COMP0178")]
    if (p / "sql_practice.py").exists()), None)
if practice_folder is None:
    raise FileNotFoundError("Keep sql_practice.py in the COMP0178 folder.")
sys.path.insert(0, str(practice_folder))
from sql_practice import Practice
lab = Practice()
lab.register(get_ipython())

In [ ]:
%%sql
SELECT staffNo, salary FROM Staff WHERE salary = 30000;

In [ ]:
lab.check([("S001", 30000)])
lab.connection.execute("CREATE INDEX IF NOT EXISTS idx_staff_salary ON Staff(salary)")
plan = lab.connection.execute(
    "EXPLAIN QUERY PLAN SELECT staffNo FROM Staff WHERE salary = 30000"
).fetchall()
for row in plan:
    print(row)
assert any("idx_staff_salary" in str(row) for row in plan)
print("The example plan uses the salary index; runtime plan wording may vary.")

### Practice A — Explain the access choice

For each query choose a sorted index, hashing, or explain why a scan may be sensible:

1. Find one known staff number.
2. List all properties with rent between 700 and 1,400 in rent order.
3. Return nearly every row of a small table.
4. Why might a dirty page cost more to evict than a clean page?

In [ ]:
answer_a = ""
print(answer_a if answer_a.strip() else "Write your reasoning in answer_a before reading the solutions.")

<details><summary>Worked answers</summary>

1. Either an equality-capable tree index or suitable hashing; hashing is attractive for exact keys. State available structures.
2. A sorted rent index supports the range and order; a hash on rent does not directly support this traversal.
3. A scan can avoid index traversal and many individual data lookups; an index is not automatically cheaper.
4. A dirty page has changes requiring preservation/write-back under the recovery policy; a clean cached copy can simply be discarded.

</details>

## 15. Exact past-paper connections

- **[2023 Q5(b), 5 marks](../Exampapers/COMP0178A7PE.pdf#page=5):** whether hashing table keys can quickly determine fragmentation. **Worked short answer:** true—map each key deterministically to a fragment, e.g. `h(userID) mod N`. This makes routing easy and can distribute records; skew and poor query locality remain possible. Hash partitioning is a use of hashing, not the same thing as an index.
- **[2024 Q5(b), 5 marks](../Exampapers/COMP0178A7PF.pdf#page=5):** give two uses of hashing in databases. Two valid explained examples are an equality hash index (key → bucket) and hash fragmentation (key → owning fragment). A hash join is another later application: build buckets on a join key, then probe with the other input.
- **2023 Q2(d), [PDF page 3](../Exampapers/COMP0178A7PE.pdf#page=3); 2024 Q2(c), [page 3](../Exampapers/COMP0178A7PF.pdf#page=3); 2025 Q2(d), [pages 3–4](../Exampapers/COMP0178A7PG.pdf#page=3):** query costs explicitly assume **no indexes** and **one tuple per read**. Use those exam assumptions rather than this lecture's ordinary multi-record-page example. Lesson 3b works through the SQL/algebra/tree/cost chain.
- 2023 asks for **reads**; 2024/2025 ask for **accesses**. Separate reads and writes and state what you count. In 2024 Q2, the printed part marks add to 20 while its printed question total says 30; preserve that discrepancy.

## 16. Revision questions and answers

1. Why group records into pages? Transfers occur in chunks and multiple small records can share a transfer.
2. What does dirty mean? The buffered version differs from its persistent data-page copy.
3. Does flushing necessarily empty the cache? No.
4. Can a primary key and a clustered key differ? Yes; logical identity and physical ordering are separate.
5. Why can a sparse index work? Ordered data lets a nearby entry direct us to the relevant range.
6. Why are B+ trees shallow? Each page-sized node has high fan-out.
7. Does every B+ tree leaf hold full rows? No; that is a clustered variant, while others hold references.
8. What is a collision? Different keys map to the same bucket.
9. What does hashing not provide naturally? Sorted/range traversal.
10. Does a page read always equal one tuple read? No; use the problem's stated model.

**Next:** Lesson 3b describes logical operations on tables. Storage explains their eventual physical cost; the algebra itself does not dictate one physical execution algorithm.